In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import copy
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn

DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
file_path = DATA_DIR / "GM01.mat"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
mat_data = sio.loadmat(file_path)

# Print the keys to see how the author named the variables
print("Keys inside GM01.mat:")
for key in mat_data.keys():
    if not key.startswith('__'):  # Ignore Python/MATLAB metadata
        data_type = type(mat_data[key])
        shape = mat_data[key].shape if isinstance(mat_data[key], np.ndarray) else "N/A"
        print(f" - '{key}': Type {data_type}, Shape {shape}")

In [ ]:
img = mat_data["img"]   # (1243, 684, 224)
gt_map = mat_data["map"] # (1243, 684)

unique_labels, counts = np.unique(gt_map, return_counts=True)
total_pixels = gt_map.size

print("--- Class Distribution ---")
for label, count in zip(unique_labels, counts):
    pct = (count / total_pixels) * 100
    print(f"Class {label}: {count:,} pixels ({pct:.2f}%)")

# Plot Mean Spectral Signature for each class
plt.figure(figsize=(10, 5))
for label in unique_labels:
    mask = (gt_map == label)
    # Average across all spatial pixels belonging to this class
    mean_spectrum = img[mask].mean(axis=0)
    plt.plot(mean_spectrum, label=f"Class {label} (Pixels: {count:,})", linewidth=1.8)

plt.title("Mean Spectral Profile per Class (GM01)")
plt.xlabel("Spectral Band Index (0 to 223)")
plt.ylabel("Reflectance / Radiance")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
def get_enhanced_rgb(img_array, rgb_bands=[29, 19, 9]):
    """
    Extracts and enhances RGB bands from the raw hyperspectral image.
    Red, Green, Blue bands (approx ~650nm, ~550nm, ~480nm in AVIRIS)
    """
    rgb_raw = img_array[:, :, rgb_bands].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    
    rgb_enhanced = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_enhanced[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
    return rgb_enhanced

In [ ]:
img = mat_data["img"]   # (1243, 684, 224)
gt_map = mat_data["map"] # (1243, 684)

rgb_enhanced = get_enhanced_rgb(img)

# Plot side-by-side
fig, axs = plt.subplots(1, 2, figsize=(14, 7))
axs[0].imshow(rgb_enhanced)
axs[0].set_title("Enhanced RGB Visual (Bands 29, 19, 9)")
axs[0].axis("off")

im = axs[1].imshow(gt_map, cmap="inferno")
axs[1].set_title("Ground Truth Oil Spill Mask")
axs[1].axis("off")
fig.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# THIS IS JUST A DUMMY CODE, NEED TO CHANGE.
# ONCE YOU FIT THE PCA ON ONLY THE TRAINING DATA, PASS THAT FITTED MODEL INTO THE DATASET CLASS. 
# IT MUST BE APPLIED DURING THE BUILDING OF THE .NPY FILES IN THE NEXT CELL SO THAT ALL DATA (TRAIN, VAL, AND TEST) IS DIMENSIONALITY REDUCED BEFORE BEING CACHED.

from sklearn.decomposition import PCA

def preprocess_hyperspectral_data(img_array, num_components=30):
    """
    Dummy pipeline for removing noisy bands and reducing dimensionality.
    Run this on 'img' after loading from sio.loadmat(), before saving to cache.
    """
    # Noisy Band Removal (Specify indices of known noisy/water-absorption bands)
    noisy_bands = list(range(100, 115)) + list(range(150, 170))
    valid_bands = [b for b in range(img_array.shape[-1]) if b not in noisy_bands]
    
    img_clean = img_array[:, :, valid_bands]

    # PCA Dimensionality Reduction
    h, w, c = img_clean.shape
    img_reshaped = img_clean.reshape(h * w, c)
    
    pca = PCA(n_components=num_components)
    img_pca = pca.fit_transform(img_reshaped)
    
    # Reshape back to spatial dimensions
    return img_pca.reshape(h, w, num_components)

In [ ]:
import bisect

class HyperspectralDataset(Dataset):
    def __init__(self, file_paths, cache_dir, is_train=True):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.is_train = is_train
        
        self.cumulative_sizes = []
        self.pixel_mmaps = []
        self.label_mmaps = []
        
        total_samples = 0
        
        print(f"Initializing {'Training' if is_train else 'Testing'} Dataset...")
        for file_path in file_paths:
            file_stem = file_path.stem
            pixel_cache = self.cache_dir / f"{file_stem}_{'train' if is_train else 'test'}_pixels.npy"
            label_cache = self.cache_dir / f"{file_stem}_{'train' if is_train else 'test'}_labels.npy"
            
            if not (pixel_cache.exists() and label_cache.exists()):
                print(f"Caching {file_stem} to disk...")
                mat_data = sio.loadmat(file_path)
                img = mat_data["img"]
                gt_map = mat_data["map"]
                
                img_flat = img.reshape(-1, img.shape[-1]).astype(np.float32)
                gt_flat = gt_map.reshape(-1).astype(np.int64)
                
                if is_train:
                    water_mask = (gt_flat == 0)
                    img_flat = img_flat[water_mask]
                    gt_flat = gt_flat[water_mask]
                    
                np.save(pixel_cache, img_flat)
                np.save(label_cache, gt_flat)
            
            p_mmap = np.load(pixel_cache, mmap_mode='r')
            l_mmap = np.load(label_cache, mmap_mode='r')
            
            self.pixel_mmaps.append(p_mmap)
            self.label_mmaps.append(l_mmap)
            
            total_samples += len(l_mmap)
            self.cumulative_sizes.append(total_samples)

    def __len__(self):
        return self.cumulative_sizes[-1] if self.cumulative_sizes else 0

    # Maps a global dataset index to the correct file chunk and local offset
    def __getitem__(self, idx):
        file_idx = bisect.bisect_right(self.cumulative_sizes, idx) # binary search
        
        local_idx = idx if file_idx == 0 else idx - self.cumulative_sizes[file_idx - 1]
            
        pixel = self.pixel_mmaps[file_idx][local_idx]
        label = self.label_mmaps[file_idx][local_idx]
        
        x_tensor = torch.tensor(pixel, dtype=torch.float32).unsqueeze(0)
        y_tensor = torch.tensor(label, dtype=torch.long)
        
        return x_tensor, y_tensor


DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
CACHE_DIR = DATA_DIR / "cache"

all_files = sorted(list(DATA_DIR.glob("*.mat")))

train_files, val_files, test_files = [], [], []

# test files are GM01 and GM02, validation file can be GM03, rest are training files
for f in all_files:
    if f.stem in ["GM01", "GM02"]:
        test_files.append(f)
    elif f.stem == "GM03":
        val_files.append(f)
    else:
        train_files.append(f)

train_dataset = HyperspectralDataset(train_files, cache_dir=CACHE_DIR, is_train=True)
val_dataset = HyperspectralDataset(val_files, cache_dir=CACHE_DIR, is_train=False)
test_dataset = HyperspectralDataset(test_files, cache_dir=CACHE_DIR, is_train=False)

BATCH_SIZE = 1024

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from torchinfo import summary

class Dummy1DAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2) 
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2), 
            nn.Conv1d(in_channels=16, out_channels=1, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Dummy1DAutoencoder().to(device)
summary(model, input_size=[32, 1, 224])

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, f1_score

def calculate_metrics(reconstruction_errors, true_labels, threshold):
    predictions = (reconstruction_errors > threshold).int().cpu().numpy()
    actuals = (true_labels > 0).int().cpu().numpy()
    scores = reconstruction_errors.cpu().numpy()

    try:
        auc = roc_auc_score(actuals, scores)
    except ValueError:
        auc = 0.0 # Handles edge cases where a batch might only have one class
        
    precision = precision_score(actuals, predictions, zero_division=0)
    recall = recall_score(actuals, predictions, zero_division=0)
    accuracy = accuracy_score(actuals, predictions)
    f1 = f1_score(actuals, predictions, zero_division=0)

    return {"AUC": auc, "Precision": precision, "Recall": recall, "Accuracy": accuracy, "F1-Score": f1}

def train_step(model: nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: nn.Module, optimizer: torch.optim.Optimizer, device):
    model.train()
    train_loss = 0 

    for batch, (X, _) in enumerate(dataloader):
        X = X.to(device)

        reconstructed = model(X)
        loss = loss_fn(reconstructed, X)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss / len(dataloader)


def test_step(model: nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: nn.Module, device, threshold: float):
    model.eval()
    test_loss = 0
    
    all_errors = []
    all_labels = []

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X = X.to(device)
            y = y.to(device) # We now load the ground truth labels
            
            reconstructed = model(X)
            
            # Calculate standard batch loss
            loss = loss_fn(reconstructed, X)
            test_loss += loss.item()
            
            # Calculate MSE per pixel to evaluate the threshold
            # X shape is (Batch, 1, 224). We take the mean across the spectral dimension.
            pixel_errors = torch.mean((reconstructed - X)**2, dim=[1, 2])
            
            all_errors.append(pixel_errors)
            all_labels.append(y)
            
    # Concatenate all batches for a full epoch evaluation
    all_errors = torch.cat(all_errors)
    all_labels = torch.cat(all_labels)
    
    rates = calculate_metrics(all_errors, all_labels, threshold)

    return (test_loss / len(dataloader)), rates

In [ ]:
from tqdm.auto import tqdm
import copy

def train(model: nn.Module, train_dataloader: torch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module, epochs: int, threshold: float, device):
          
    results = {"train_loss": [], "test_loss": [], "TPR": [], "TNR": [], "FPR": [], "FNR": []}
    
    best_test_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in tqdm(range(epochs)):
        train_loss = train_step(model=model,
                                dataloader=train_dataloader,
                                loss_fn=loss_fn,
                                optimizer=optimizer,
                                device=device)
                                
        test_loss, rates = test_step(model=model,
                                     dataloader=test_dataloader,
                                     loss_fn=loss_fn,
                                     device=device,
                                     threshold=threshold)

        print(f"Epoch: {epoch} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}") 
        print(f"         AUC: {rates['AUC']:.4f} | Precision: {rates['Precision']:.4f} | Recall: {rates['Recall']:.4f} | Acc: {rates['Accuracy']:.4f} | F1: {rates['F1-Score']:.4f}")

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model_wts = copy.deepcopy(model.state_dict())

        results["train_loss"].append(train_loss)
        results["test_loss"].append(test_loss)
        results["TPR"].append(rates["TPR"])
        results["TNR"].append(rates["TNR"])
        results["FPR"].append(rates["FPR"])
        results["FNR"].append(rates["FNR"])

    model.load_state_dict(best_model_wts)
    return model, results

In [ ]:
NUM_EPOCHS = 1
LEARNING_RATE = 0.001
THRESHOLD = 50.0

loss_fn = nn.MSELoss()

optimizer = torch.optim.AdamW(params=model.parameters(), lr=LEARNING_RATE)

print(f"Starting training on {device} for {NUM_EPOCHS} epochs...")

best_model, results = train(model=model,
                            train_dataloader=train_dataloader,
                            test_dataloader=test_dataloader,
                            optimizer=optimizer,
                            loss_fn=loss_fn,
                            epochs=NUM_EPOCHS,
                            threshold=THRESHOLD,
                            device=device)

print("Training complete! The best model weights have been restored.")

In [ ]:
def visualize_reconstructions(model, file_path, device):
    mat_data = sio.loadmat(file_path)
    img = mat_data["img"]
    gt_map = mat_data["map"]
    
    img_flat = img.reshape(-1, img.shape[-1]).astype(np.float32)
    gt_flat = gt_map.reshape(-1).astype(np.int64)
    
    water_indices = np.where(gt_flat == 0)[0]
    oil_indices = np.where(gt_flat > 0)[0]
        
    water_pixel = img_flat[np.random.choice(water_indices)]
    oil_pixel = img_flat[np.random.choice(oil_indices)]
    
    water_tensor = torch.tensor(water_pixel).unsqueeze(0).unsqueeze(0).to(device)
    oil_tensor = torch.tensor(oil_pixel).unsqueeze(0).unsqueeze(0).to(device)
    
    model.eval()
    with torch.inference_mode():
        water_recon = model(water_tensor).squeeze().cpu().numpy()
        oil_recon = model(oil_tensor).squeeze().cpu().numpy()
        
    # Mean Squared Error
    water_mse = np.mean((water_pixel - water_recon)**2)
    oil_mse = np.mean((oil_pixel - oil_recon)**2)
    
    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    
    axs[0].plot(water_pixel, label="Original Water Signature", color="blue", linewidth=1.5)
    axs[0].plot(water_recon, label="Autoencoder Reconstruction", color="green", linestyle="--", linewidth=1.5)
    axs[0].set_title(f"Normal Water Pixel (MSE: {water_mse:.2f})")
    axs[0].set_xlabel("Spectral Band")
    axs[0].set_ylabel("Radiance")
    axs[0].legend()
    axs[0].grid(True)
    
    axs[1].plot(oil_pixel, label="Original Oil Signature", color="orange", linewidth=1.5)
    axs[1].plot(oil_recon, label="Autoencoder Reconstruction", color="red", linestyle="--", linewidth=1.5)
    axs[1].set_title(f"Anomalous Oil Pixel (MSE: {oil_mse:.2f})")
    axs[1].set_xlabel("Spectral Band")
    axs[1].set_ylabel("Radiance")
    axs[1].legend()
    axs[1].grid(True)
    
    plt.tight_layout()
    plt.show()

# test_file_path = test_files[0]
# visualize_reconstructions(best_model, test_file_path, device)

In [ ]:
def plot_gm01_predictions(model, file_path, threshold, device):
    mat_data = sio.loadmat(file_path)
    img = mat_data["img"]
    gt_map = mat_data["map"]
    
    h, w, c = img.shape
    img_flat = img.reshape(-1, c).astype(np.float32)
    
    # Process predictions in batches
    model.eval()
    errors = []
    batch_size = 2048
    
    with torch.inference_mode():
        for i in range(0, len(img_flat), batch_size):
            batch = torch.tensor(img_flat[i:i+batch_size]).unsqueeze(1).to(device)
            recon = model(batch)
            mse = torch.mean((recon - batch)**2, dim=[1, 2]).cpu().numpy()
            errors.extend(mse)
            
    errors = np.array(errors).reshape(h, w)
    predictions = (errors > threshold).astype(int)
    
    # RGB Enhancement using bands 29, 19, 9
    rgb_bands = [29, 19, 9]
    rgb_enhanced = get_enhanced_rgb(img)

    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    axs[0].imshow(rgb_enhanced)
    axs[0].set_title("Actual Data (RGB Enhanced)")
    axs[0].axis("off")
    
    axs[1].imshow(gt_map, cmap="inferno")
    axs[1].set_title("Ground Truth Mask")
    axs[1].axis("off")
    
    axs[2].imshow(predictions, cmap="inferno")
    axs[2].set_title(f"Model Prediction (Threshold={threshold})")
    axs[2].axis("off")
    
    plt.tight_layout()
    plt.show()

gm01_path = DATA_DIR / "GM01.mat"
plot_gm01_predictions(best_model, gm01_path, THRESHOLD, device)